In [2]:
import os
import random
import time
import joblib
import numpy as np
import polars as pl
import psutil
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from tensorflow import keras
# from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, roc_curve, auc
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

# ART

from art.attacks.evasion import ProjectedGradientDescent, FastGradientMethod
from art.estimators.classification import TensorFlowV2Classifier, SklearnClassifier, XGBoostClassifier, LightGBMClassifier, CatBoostARTClassifier, KerasClassifier


In [3]:
# Carga Dataset UNSW-NB15

path_train = "../../DATASETS/dataSets_Reducidos/nusw-nb15/datos_train_NUSW_redux.csv"
path_test  = "../../DATASETS/dataSets_Reducidos/nusw-nb15/datos_test_NUSW_redux.csv"
TARGET_COL = "attack_cat"

df_train = pl.read_csv(path_train)
df_test  = pl.read_csv(path_test)

y_train = (
    df_train.select(
        pl.when(pl.col(TARGET_COL).str.strip_chars() == "Normal")
        .then(1)
        .otherwise(-1)
        .alias("label")
    )
    .to_series()
    .cast(pl.Int8)
)

y_test = (
    df_test.select(
        pl.when(pl.col(TARGET_COL).str.strip_chars() == "Normal")
        .then(1)
        .otherwise(-1)
        .alias("label")
    )
    .to_series()
    .cast(pl.Int8)
)

x_train = df_train.drop(TARGET_COL)
x_test  = df_test.drop(TARGET_COL)

X_full_train = x_train.to_numpy().astype(np.float32)
X_test_np = x_test.to_numpy().astype(np.float32)
y_full_train = y_train.to_numpy()
y_test_np = y_test.to_numpy()

y_full_train_01 = ((y_full_train + 1) // 2).astype(np.int8)
y_test_np01 = ((y_test_np + 1) // 2).astype(np.int8)

# Preparamos una sola vez las distintas vistas del dataset.
# Mantenemos el espacio original para restricciones tabulares y transferencia
# entre modelos que no comparten el mismo preprocesado.
mlp_scaler = StandardScaler()
X_train_scaled_mlp = mlp_scaler.fit_transform(X_full_train).astype(np.float32)
X_test_scaled_mlp = mlp_scaler.transform(X_test_np).astype(np.float32)

cnn_scaler = MinMaxScaler()
X_train_scaled_cnn = cnn_scaler.fit_transform(X_full_train).astype(np.float32)
X_test_scaled_cnn = cnn_scaler.transform(X_test_np).astype(np.float32)

DATASET_VIEWS = {
    "raw": {"train": X_full_train, "test": X_test_np},
    "standard": {"train": X_train_scaled_mlp, "test": X_test_scaled_mlp},
    "minmax": {"train": X_train_scaled_cnn, "test": X_test_scaled_cnn},
}

print(f"Train: {X_full_train.shape} | Test: {X_test_np.shape}")
print("Tras convertir -1/1 a 0/1, la clase 0 corresponde a Ataque y la clase 1 a Normal.")
print("Vistas disponibles:", ", ".join(DATASET_VIEWS.keys()))

HAS_GPU = len(tf.config.list_physical_devices("GPU")) > 0
TRAIN_DEVICE = "/GPU:0" if HAS_GPU else "/CPU:0"
INFER_DEVICE = "/CPU:0"

SEED = 42
MODEL_DIR = os.path.join("model", "unsw.nb15")

random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)


Train: (175341, 12) | Test: (82332, 12)
Tras convertir -1/1 a 0/1, la clase 0 corresponde a Ataque y la clase 1 a Normal.
Vistas disponibles: raw, standard, minmax


In [4]:
# Modelos ganadores de UNSW-NB15

RF_CONFIG = {"n_estimators": 50, "max_depth": 23}
XGB_CONFIG = {"n_estimators": 200, "max_depth": 11, "learning_rate": 0.1}
LGBM_CONFIG = {"n_estimators": 100, "num_leaves": 145, "max_depth": 12, "learning_rate": 0.1}
CATBOOST_CONFIG = {"iterations": 500, "depth": 10, "learning_rate": 0.1}

SVM_C = 0.000187
MLP_INPUT_DIM = X_full_train.shape[1]
CNN1D_CONFIG = {"nf": 64, "k": 5, "d": 48}

def build_mlp_model(input_dim):
    model = keras.Sequential([
        keras.layers.InputLayer(input_shape=(input_dim,)),
        keras.layers.Dense(96, activation="relu"),
        keras.layers.Dense(32, activation="relu"),
        keras.layers.Dense(64, activation="relu"),
        keras.layers.Dense(1, activation="sigmoid")
    ])
    model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    return model

DEFAULT_CNN_DROPOUT = 0.2

def build_cnn1d_model(n_features, n_filters, kernel_size, dense_units, dropout_rate=DEFAULT_CNN_DROPOUT):
    model = keras.Sequential([
        keras.layers.Input(shape=(n_features, 1)),
        keras.layers.Conv1D(filters=n_filters, kernel_size=kernel_size, padding="same", activation="relu"),
        keras.layers.MaxPooling1D(pool_size=2),
        keras.layers.Conv1D(filters=max(16, n_filters // 2), kernel_size=kernel_size, padding="same", activation="relu"),
        keras.layers.GlobalMaxPooling1D(),
        keras.layers.Dense(dense_units, activation="relu"),
        keras.layers.Dropout(dropout_rate),
        keras.layers.Dense(1, activation="sigmoid")
    ])
    model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    return model

def clone_keras_model_to_cpu(builder_fn, trained_model, *builder_args):
    with tf.device(INFER_DEVICE):
        cpu_model = builder_fn(*builder_args)
        cpu_model.set_weights(trained_model.get_weights())
    return cpu_model

In [ ]:
# Carga de modelos entrenados para UNSW-NB15

MODEL_PATHS = {
    "rf": os.path.join(MODEL_DIR, "rf.joblib"),
    "xgb": os.path.join(MODEL_DIR, "xgb.joblib"),
    "lgbm": os.path.join(MODEL_DIR, "lgbm.joblib"),
    "catboost": os.path.join(MODEL_DIR, "catboost.joblib"),
    "svm": os.path.join(MODEL_DIR, "svm.joblib"),
    "mlp": os.path.join(MODEL_DIR, "mlp.joblib"),
    "cnn": os.path.join(MODEL_DIR, "cnn.joblib"),
}

def load_model(model_key):
    model_path = MODEL_PATHS[model_key]
    print(f"Cargando {model_key} desde {model_path}...")
    return joblib.load(model_path)

rf_model = load_model("rf")
xgb_model = load_model("xgb")
lgbm_model = load_model("lgbm")
cat_model = load_model("catboost")


In [ ]:
# Carga de SVM y modelos neuronales

svm_model = load_model("svm")
mlp_model = load_model("mlp")

X_train_tabular_cnn = X_train_scaled_cnn.reshape(X_train_scaled_cnn.shape[0], X_train_scaled_cnn.shape[1], 1)
X_test_tabular_cnn = X_test_scaled_cnn.reshape(X_test_scaled_cnn.shape[0], X_test_scaled_cnn.shape[1], 1)

cnn_model = load_model("cnn")


Una vez que los modelos ya están entrenados, ya podemos entrar en la fase de Evaluación Adversaria. Primero vamos a ver que nuestro modelo recién entrenado rinda bien con x_test limpio

In [7]:
# Evaluamos en test MLP

mlp_model_cpu = clone_keras_model_to_cpu(build_mlp_model, mlp_model, MLP_INPUT_DIM)
def mlp_predict_labels(X):
    with tf.device(INFER_DEVICE):
        y_prob = mlp_model_cpu.predict(X, batch_size=4096, verbose=0).ravel()
    return (y_prob > 0.5).astype(np.int8)
def mlp_predict_scores(X):
    with tf.device(INFER_DEVICE):
        return mlp_model_cpu.predict(X, batch_size=4096, verbose=0).ravel()
    
def cnn_predict_labels(X):
    with tf.device(INFER_DEVICE):
        y_prob = cnn_model.predict(X, batch_size=4096, verbose=0).ravel()
    return (y_prob > 0.5).astype(np.int8)

# F1-score de referencia en test para MLP
y_test_pred_mlp = mlp_predict_labels(X_test_scaled_mlp)
mlp_f1 = f1_score(y_test_np01, y_test_pred_mlp, pos_label=0)
print(f"\nMLP F1-score en test: {mlp_f1:.4f}")


MLP F1-score en test: 0.8497


In [8]:
# Extraemos restricciones tabulares a partir del train original
feature_names = x_train.columns

tabular_constraints_df = pl.DataFrame({
    "feature": feature_names,
    "min": x_train.min().row(0),
    "max": x_train.max().row(0),
})

feature_mins = tabular_constraints_df["min"].to_numpy().astype(np.float32)
feature_maxs = tabular_constraints_df["max"].to_numpy().astype(np.float32)

tabular_constraints = {
    feature: {"min": float(min_val), "max": float(max_val)}
    for feature, min_val, max_val in zip(feature_names, feature_mins, feature_maxs)
}

# 1. Restricciones para modelos que trabajan en espacio original
clip_values_raw = (feature_mins, feature_maxs)

# 2. Restricciones para el MLP, que trabaja con StandardScaler
feature_mins_mlp = mlp_scaler.transform(feature_mins.reshape(1, -1)).ravel().astype(np.float32)
feature_maxs_mlp = mlp_scaler.transform(feature_maxs.reshape(1, -1)).ravel().astype(np.float32)
clip_values_mlp = (feature_mins_mlp, feature_maxs_mlp)

# 3. Restricciones para la CNN, que trabaja con MinMaxScaler
feature_mins_cnn = cnn_scaler.transform(feature_mins.reshape(1, -1)).reshape(-1, 1).astype(np.float32)
feature_maxs_cnn = cnn_scaler.transform(feature_maxs.reshape(1, -1)).reshape(-1, 1).astype(np.float32)
clip_values_cnn = (feature_mins_cnn, feature_maxs_cnn)

print("Restricciones tabulares extraidas para UNSW-NB15:")
display(tabular_constraints_df)

print(clip_values_mlp)

Restricciones tabulares extraidas para UNSW-NB15:


feature,min,max
str,f64,f64
"""dur""",0.0,59.999989
"""sinpkt""",0.0,84371.496
"""dinpkt""",0.0,56716.824
"""spkts""",1.0,9616.0
"""dpkts""",0.0,10974.0
…,…,…
"""rate""",0.0,1.0000e6
"""smean""",28.0,1504.0
"""dmean""",0.0,1458.0


(array([-0.20977475, -0.1361428 , -0.08937003, -0.14098223, -0.17204736,
       -0.05044967, -0.10392289, -0.57681924, -0.5313342 , -0.48070285,
       -0.48434597, -0.50301373], dtype=float32), array([  9.049153 ,  11.513798 ,  57.369225 ,  70.09933  ,  99.358185 ,
        74.135994 , 101.91599  ,   5.469112 ,   6.6800365,   5.1635404,
        47.911243 ,  37.04389  ], dtype=float32))


In [9]:
# Hay que tener en cuenta que si tuviéramos columnas One Hot Encoding, habría que asegurarse de que las restricciones 
# se apliquen correctamente y que ART no me genere un TCP = 0.62, ya que no tiene sentido que un valor One Hot Encoding tenga un valor intermedio entre 0 y 1. 
# En ese caso habría que indicarle a ART que esas columnas solo pueden tomar valores 0 o 1, y no valores continuos.

# ==========================================
# FASE 1. ENVOLVER EL MODELO (Caja Blanca)
# ==========================================

print("Envolviendo el modelo MLP en ART con restricciones tabulares...")

clasificador_art_mlp = KerasClassifier(
    model=mlp_model_cpu, 
    clip_values=clip_values_mlp, 
    use_logits=False
)

print("Envolviendo el modelo CNN en ART con restricciones tabulares...")

cnn_model_cpu = clone_keras_model_to_cpu(
    build_cnn1d_model,
    cnn_model,
    X_train_tabular_cnn.shape[1],
    CNN1D_CONFIG["nf"],
    CNN1D_CONFIG["k"],
    CNN1D_CONFIG["d"]
)

clasificador_art_cnn = KerasClassifier(
    model=cnn_model_cpu,
    clip_values=clip_values_cnn,
    use_logits=False
)


Envolviendo el modelo MLP en ART con restricciones tabulares...
Envolviendo el modelo CNN en ART con restricciones tabulares...


In [10]:
# ========================================================
# FASE 4. LANZAR ATAQUE FGSM PARA VARIOS LIMITES DE PERTURBACION
# ========================================================

EPS_VALUES = [0.01, 0.02, 0.03, 0.04, 0.05, 0.06, 0.07, 0.08, 0.09, 0.1, 0.2, 0.3, 0.4, 0.5]

MODELOS_TABLA = ["RF", "XGB", "LGBM", "CatBoost", "SVM", "MLP", "CNN"]

modelos_arbol = {
    "RF": rf_model,
    "XGB": xgb_model,
    "LGBM": lgbm_model,
    "CatBoost": cat_model,
}

modelos_clasicos = {
    "RF": rf_model,
    "XGB": xgb_model,
    "LGBM": lgbm_model,
    "CatBoost": cat_model,
    "SVM": svm_model
}


In [11]:
feature_ranges_mlp = clip_values_mlp[1] - clip_values_mlp[0]

resultados_fgsm_mlp = []
x_test_mlp_attack = X_test_scaled_mlp.astype(np.float32)

print("Generando FGSM sobre MLP para varios limites de perturbacion...")

for eps_base in EPS_VALUES:
    eps_vector_mlp = (eps_base * feature_ranges_mlp).astype(np.float32)
    eps_step_vector_mlp = np.where(eps_vector_mlp > 0, eps_vector_mlp / 4.0, 0.0).astype(np.float32)

    ataque_fgsm_mlp = FastGradientMethod(
        estimator=clasificador_art_mlp,
        eps=eps_vector_mlp,
        eps_step=eps_step_vector_mlp,
        batch_size=128,
    )

    # Generamos adversarios en el espacio del MLP
    x_test_fgsm_mlp = ataque_fgsm_mlp.generate(x=x_test_mlp_attack)

    # Preparamos las distintas transformaciones para evaluar los modelos en sus respectivos espacios
    x_test_fgsm_mlp_std = x_test_fgsm_mlp.astype(np.float32)
    x_test_fgsm_mlp_raw = mlp_scaler.inverse_transform(x_test_fgsm_mlp_std).astype(np.float32)
    x_test_fgsm_mlp_minmax = cnn_scaler.transform(x_test_fgsm_mlp_raw).astype(np.float32)
    x_test_fgsm_mlp_cnn = x_test_fgsm_mlp_minmax.reshape(
        x_test_fgsm_mlp_minmax.shape[0],
        x_test_fgsm_mlp_minmax.shape[1],
        1
    )

# Inicializar la fila con el valor actual del límite
    fila_resultados = {"Limite perturbacion": eps_base}

# 4. Bucle UNIFICADO para evaluar todos los modelos de la tabla
    for nombre_modelo in MODELOS_TABLA:
        if nombre_modelo in ["RF", "XGB", "LGBM", "CatBoost"]:
            # Modelos de árboles usan datos RAW
            y_pred = modelos_clasicos[nombre_modelo].predict(x_test_fgsm_mlp_raw)
        
        elif nombre_modelo == "SVM":
            # SVM usa datos estandarizados
            y_pred = modelos_clasicos[nombre_modelo].predict(x_test_fgsm_mlp_std)
            
        elif nombre_modelo == "MLP":
            # MLP requiere su función auxiliar
            y_pred = mlp_predict_labels(x_test_fgsm_mlp_std)
            
        elif nombre_modelo == "CNN":
            # CNN requiere su función auxiliar y datos minmax redimensionados
            y_pred = cnn_predict_labels(x_test_fgsm_mlp_cnn)
            
        # Calcular y guardar el F1-score directamente
        fila_resultados[nombre_modelo] = f1_score(y_test_np01, y_pred, pos_label=0)

    resultados_fgsm_mlp.append(fila_resultados)

print("Evaluacion FGSM asociada al MLP completada.\n")

Generando FGSM sobre MLP para varios limites de perturbacion...


/home/placivm_tfg/PLACI_TFG/.venv_tfg/lib/python3.11/site-packages/xgboost/core.py:751: UserWarning: [11:49:13] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)
/home/placivm_tfg/PLACI_TFG/.venv_tfg/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/placivm_tfg/PLACI_TFG/.venv_tfg/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
 

Evaluacion FGSM asociada al MLP completada.



In [12]:
feature_ranges_cnn = clip_values_cnn[1] - clip_values_cnn[0]

resultados_fgsm_cnn = []
x_test_cnn_attack = X_test_tabular_cnn.astype(np.float32)

print("Generando FGSM sobre CNN para varios limites de perturbacion...")

for eps_base in EPS_VALUES:
    eps_vector_cnn = (eps_base * feature_ranges_cnn).astype(np.float32)
    eps_step_vector_cnn = np.where(eps_vector_cnn > 0, eps_vector_cnn / 4.0, 0.0).astype(np.float32)

    ataque_fgsm_cnn = FastGradientMethod(
        estimator=clasificador_art_cnn,
        eps=eps_vector_cnn,
        eps_step=eps_step_vector_cnn,
        batch_size=128,
    )

    with tf.device(INFER_DEVICE):
        x_test_fgsm_cnn = ataque_fgsm_cnn.generate(x=x_test_cnn_attack)

    x_test_fgsm_cnn_cnn = x_test_fgsm_cnn.astype(np.float32)
    x_test_fgsm_cnn_minmax = x_test_fgsm_cnn_cnn.reshape(
        x_test_fgsm_cnn_cnn.shape[0],
        x_test_fgsm_cnn_cnn.shape[1]
    ).astype(np.float32)
    x_test_fgsm_cnn_raw = cnn_scaler.inverse_transform(x_test_fgsm_cnn_minmax).astype(np.float32)
    x_test_fgsm_cnn_std = mlp_scaler.transform(x_test_fgsm_cnn_raw).astype(np.float32)

# Inicializar la fila con el valor actual del límite
    fila_resultados = {"Limite perturbacion": eps_base}

# 4. Bucle UNIFICADO para evaluar todos los modelos de la tabla
    for nombre_modelo in MODELOS_TABLA:
        if nombre_modelo in ["RF", "XGB", "LGBM", "CatBoost"]:
            # Modelos de árboles usan datos RAW
            y_pred = modelos_clasicos[nombre_modelo].predict(x_test_fgsm_cnn_raw)
        
        elif nombre_modelo == "SVM":
            # SVM usa datos estandarizados
            y_pred = modelos_clasicos[nombre_modelo].predict(x_test_fgsm_cnn_std)
            
        elif nombre_modelo == "MLP":
            # MLP requiere su función auxiliar
            y_pred = mlp_predict_labels(x_test_fgsm_cnn_std)
            
        elif nombre_modelo == "CNN":
            # CNN requiere su función auxiliar y datos minmax redimensionados
            y_pred = cnn_predict_labels(x_test_fgsm_cnn_cnn)
            
        # Calcular y guardar el F1-score directamente
        fila_resultados[nombre_modelo] = f1_score(y_test_np01, y_pred, pos_label=0)

    resultados_fgsm_cnn.append(fila_resultados)

print("Evaluacion FGSM asociada a la CNN completada.\n")

Generando FGSM sobre CNN para varios limites de perturbacion...


/home/placivm_tfg/PLACI_TFG/.venv_tfg/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/placivm_tfg/PLACI_TFG/.venv_tfg/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/placivm_tfg/PLACI_TFG/.venv_tfg/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/placivm_tfg/PLACI_TFG/.venv_tfg/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/placivm_tfg/PLACI_TFG/.venv_tfg/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have va

Evaluacion FGSM asociada a la CNN completada.



In [13]:
import pandas as pd

tabla_f1_fgsm_mlp = pd.DataFrame(resultados_fgsm_mlp).set_index("Limite perturbacion")
tabla_f1_fgsm_mlp = tabla_f1_fgsm_mlp[MODELOS_TABLA].round(4)

tabla_f1_fgsm_cnn = pd.DataFrame(resultados_fgsm_cnn).set_index("Limite perturbacion")
tabla_f1_fgsm_cnn = tabla_f1_fgsm_cnn[MODELOS_TABLA].round(4)

print("Tabla FGSM asociada a la perturbacion generada sobre el MLP")
display(tabla_f1_fgsm_mlp)

print("Tabla FGSM asociada a la perturbacion generada sobre la CNN")
display(tabla_f1_fgsm_cnn)

Tabla FGSM asociada a la perturbacion generada sobre el MLP


,RF,XGB,LGBM,CatBoost,SVM,MLP,CNN
Limite perturbacion,,,,,,,
0.01,0.7319,0.6845,0.6896,0.7077,0.6881,0.6934,0.7977
0.02,0.7148,0.6860,0.6457,0.7055,0.6807,0.6211,0.6945
0.03,0.7022,0.6825,0.6531,0.7037,0.5906,0.5793,0.6912
0.04,0.7012,0.6269,0.6554,0.7010,0.5116,0.5740,0.6902
0.05,0.6999,0.5908,0.6685,0.7252,0.4765,0.5757,0.6870
0.06,0.6965,0.5895,0.6429,0.7002,0.4144,0.5808,0.6860
0.07,0.6986,0.5995,0.6404,0.7300,0.4125,0.5786,0.6836
0.08,0.6985,0.6517,0.6444,0.7222,0.4126,0.5781,0.6938
0.09,0.7118,0.6705,0.6062,0.7308,0.3978,0.5763,0.6937


Tabla FGSM asociada a la perturbacion generada sobre la CNN


,RF,XGB,LGBM,CatBoost,SVM,MLP,CNN
Limite perturbacion,,,,,,,
0.01,0.6571,0.7536,0.7327,0.6727,0.7260,0.6298,0.7737
0.02,0.6653,0.7365,0.7183,0.7027,0.7246,0.6424,0.7147
0.03,0.6842,0.7550,0.7451,0.7205,0.6808,0.6447,0.7158
0.04,0.6845,0.4327,0.7287,0.7276,0.6728,0.6211,0.7166
0.05,0.6883,0.4367,0.7319,0.7372,0.6689,0.5954,0.7172
0.06,0.6805,0.4043,0.7317,0.7325,0.6693,0.5395,0.7177
0.07,0.6762,0.3782,0.7326,0.7131,0.6691,0.5432,0.7181
0.08,0.6783,0.4271,0.7349,0.7201,0.6647,0.5264,0.7185
0.09,0.6754,0.6262,0.7339,0.6757,0.6508,0.5082,0.7139


In [14]:
# ========================================================
# FASE 5. LANZAR ATAQUE PGD PARA VARIOS LIMITES DE PERTURBACION
# ========================================================

PGD_MAX_ITER = 20

print("Configurando PGD para varios limites de perturbacion...")
print(f"Iteraciones PGD: {PGD_MAX_ITER}")


Configurando PGD para varios limites de perturbacion...
Iteraciones PGD: 20


In [15]:
feature_ranges_mlp = clip_values_mlp[1] - clip_values_mlp[0]

resultados_pgd_mlp = []
x_test_mlp_attack = X_test_scaled_mlp.astype(np.float32)

print("Generando PGD sobre MLP para varios limites de perturbacion...")

for eps_base in EPS_VALUES:
    eps_vector_mlp = (eps_base * feature_ranges_mlp).astype(np.float32)
    eps_step_vector_mlp = np.where(eps_vector_mlp > 0, eps_vector_mlp / 4.0, 0.0).astype(np.float32)

    ataque_pgd_mlp = ProjectedGradientDescent(
        estimator=clasificador_art_mlp,
        eps=eps_vector_mlp,
        eps_step=eps_step_vector_mlp,
        max_iter=PGD_MAX_ITER,
        batch_size=128,
    )

    x_test_pgd_mlp = ataque_pgd_mlp.generate(x=x_test_mlp_attack)

    x_test_pgd_mlp_std = x_test_pgd_mlp.astype(np.float32)
    x_test_pgd_mlp_raw = mlp_scaler.inverse_transform(x_test_pgd_mlp_std).astype(np.float32)
    x_test_pgd_mlp_minmax = cnn_scaler.transform(x_test_pgd_mlp_raw).astype(np.float32)
    x_test_pgd_mlp_cnn = x_test_pgd_mlp_minmax.reshape(
        x_test_pgd_mlp_minmax.shape[0],
        x_test_pgd_mlp_minmax.shape[1],
        1
    )

    fila_resultados = {"Limite perturbacion": eps_base}

    for nombre_modelo in MODELOS_TABLA:
        if nombre_modelo in ["RF", "XGB", "LGBM", "CatBoost"]:
            y_pred = modelos_clasicos[nombre_modelo].predict(x_test_pgd_mlp_raw)
        elif nombre_modelo == "SVM":
            y_pred = modelos_clasicos[nombre_modelo].predict(x_test_pgd_mlp_std)
        elif nombre_modelo == "MLP":
            y_pred = mlp_predict_labels(x_test_pgd_mlp_std)
        elif nombre_modelo == "CNN":
            y_pred = cnn_predict_labels(x_test_pgd_mlp_cnn)

        fila_resultados[nombre_modelo] = f1_score(y_test_np01, y_pred, pos_label=0)

    resultados_pgd_mlp.append(fila_resultados)

print("Evaluacion PGD asociada al MLP completada.")


Generando PGD sobre MLP para varios limites de perturbacion...


PGD - Random Initializations: 100%|██████████| 1/1 [00:00<00:00,  6.56it/s]
/home/placivm_tfg/PLACI_TFG/.venv_tfg/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
PGD - Random Initializations: 100%|██████████| 1/1 [00:00<00:00,  6.53it/s]
/home/placivm_tfg/PLACI_TFG/.venv_tfg/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
PGD - Random Initializations: 100%|██████████| 1/1 [00:00<00:00,  6.60it/s]
/home/placivm_tfg/PLACI_TFG/.venv_tfg/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
PGD - Random Initializations: 100%|██████████| 1/1 [00:00<00:00,  6.51it/s]
/home/placivm_tfg/PLACI_TFG/.venv_tfg/lib/pyt

KeyboardInterrupt: 

In [ ]:
feature_ranges_cnn = clip_values_cnn[1] - clip_values_cnn[0]

resultados_pgd_cnn = []
x_test_cnn_attack = X_test_tabular_cnn.astype(np.float32)

print("Generando PGD sobre CNN para varios limites de perturbacion...")

for eps_base in EPS_VALUES:
    eps_vector_cnn = (eps_base * feature_ranges_cnn).astype(np.float32)
    eps_step_vector_cnn = np.where(eps_vector_cnn > 0, eps_vector_cnn / 4.0, 0.0).astype(np.float32)

    ataque_pgd_cnn = ProjectedGradientDescent(
        estimator=clasificador_art_cnn,
        eps=eps_vector_cnn,
        eps_step=eps_step_vector_cnn,
        max_iter=PGD_MAX_ITER,
        batch_size=128,
    )

    with tf.device(INFER_DEVICE):
        x_test_pgd_cnn = ataque_pgd_cnn.generate(x=x_test_cnn_attack)

    x_test_pgd_cnn_cnn = x_test_pgd_cnn.astype(np.float32)
    x_test_pgd_cnn_minmax = x_test_pgd_cnn_cnn.reshape(
        x_test_pgd_cnn_cnn.shape[0],
        x_test_pgd_cnn_cnn.shape[1]
    ).astype(np.float32)
    x_test_pgd_cnn_raw = cnn_scaler.inverse_transform(x_test_pgd_cnn_minmax).astype(np.float32)
    x_test_pgd_cnn_std = mlp_scaler.transform(x_test_pgd_cnn_raw).astype(np.float32)

    fila_resultados = {"Limite perturbacion": eps_base}

    for nombre_modelo in MODELOS_TABLA:
        if nombre_modelo in ["RF", "XGB", "LGBM", "CatBoost"]:
            y_pred = modelos_clasicos[nombre_modelo].predict(x_test_pgd_cnn_raw)
        elif nombre_modelo == "SVM":
            y_pred = modelos_clasicos[nombre_modelo].predict(x_test_pgd_cnn_std)
        elif nombre_modelo == "MLP":
            y_pred = mlp_predict_labels(x_test_pgd_cnn_std)
        elif nombre_modelo == "CNN":
            y_pred = cnn_predict_labels(x_test_pgd_cnn_cnn)

        fila_resultados[nombre_modelo] = f1_score(y_test_np01, y_pred, pos_label=0)

    resultados_pgd_cnn.append(fila_resultados)

print("Evaluacion PGD asociada a la CNN completada.")


Generando PGD sobre CNN para varios limites de perturbacion...


PGD - Random Initializations:   0%|          | 0/1 [00:00<?, ?it/s]

PGD - Random Initializations:   0%|          | 0/1 [00:00<?, ?it/s]


KeyboardInterrupt: 

In [ ]:
tabla_f1_pgd_mlp = pd.DataFrame(resultados_pgd_mlp).set_index("Limite perturbacion")
tabla_f1_pgd_mlp = tabla_f1_pgd_mlp[MODELOS_TABLA].round(4)

tabla_f1_pgd_cnn = pd.DataFrame(resultados_pgd_cnn).set_index("Limite perturbacion")
tabla_f1_pgd_cnn = tabla_f1_pgd_cnn[MODELOS_TABLA].round(4)

print("Tabla PGD asociada a la perturbacion generada sobre el MLP")
display(tabla_f1_pgd_mlp)

print("Tabla PGD asociada a la perturbacion generada sobre la CNN")
display(tabla_f1_pgd_cnn)


Tabla PGD asociada a la perturbacion generada sobre el MLP


,RF,XGB,LGBM,CatBoost,SVM,MLP,CNN
Limite perturbacion,,,,,,,
0.010,0.0772,0.1939,0.5787,0.0549,0.2080,0.0741,0.2027
0.020,0.1746,0.1609,0.5162,0.0800,0.2743,0.0581,0.0977
0.030,0.0961,0.2868,0.4786,0.0528,0.2982,0.0083,0.1200
0.040,0.1038,0.4823,0.5097,0.0106,0.2144,0.0055,0.1829
0.050,0.0381,0.3883,0.5013,0.0044,0.2050,0.0032,0.1228
0.075,0.0181,0.4033,0.5398,0.0073,0.1983,0.0017,0.0905
0.100,0.0630,0.3748,0.5686,0.0451,0.1781,0.0006,0.0365


Tabla PGD asociada a la perturbacion generada sobre la CNN


,RF,XGB,LGBM,CatBoost,SVM,MLP,CNN
Limite perturbacion,,,,,,,
0.010,0.1648,0.1907,0.5058,0.2093,0.3668,0.1681,0.0614
0.020,0.3032,0.4082,0.5417,0.1543,0.3094,0.1178,0.0503
0.030,0.1896,0.4582,0.5907,0.0869,0.2372,0.1370,0.0472
0.040,0.3250,0.3630,0.5970,0.0348,0.2490,0.1548,0.0022
0.050,0.3111,0.2736,0.5273,0.0671,0.2953,0.1433,0.0017
0.075,0.2643,0.3707,0.4890,0.0224,0.2407,0.1817,0.0058
0.100,0.3033,0.3467,0.4641,0.1019,0.4223,0.2871,0.0918
